# Encodage Catégoriel
Entrée : `metadata_features.parquet` → Sortie : `features_encodees.parquet`

- Suppression des colonnes string constantes, redondantes ou à trop haute cardinalité
- One-hot encoding des colonnes catégorielles utiles (drop_first=True)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_features.parquet')
df_enc = df.copy()

print('Dataset chargé :', df_enc.shape)

## 1. Suppression des colonnes string inutiles

In [ ]:
DROP_REDUNDANT = [
    'in.building_america_climate_zone',
    'in.cec_climate_zone',
    'in.energystar_climate_zone_2023',
    'in.ashrae_iecc_climate_zone_2004_sub_cz_split',
    'in.hvac_heating_type_and_fuel',
    'in.census_division',
    'in.census_division_recs',
    'in.geometry_building_type_acs',
    'in.location_region',
    'in.ahs_region',
    'in.iso_rto_region',
    'in.puma_metro_status',
    'in.county_metro_status',
    'in.generation_and_emissions_assessment_region',
    'in.metropolitan_and_micropolitan_statistical_area',
    'in.reeds_balancing_area',
    'in.county_and_puma',
    'in.county_name',
    'in.county',
    'in.puma',
    'in.city',
    'in.state',
    'in.weather_file_city',
    'in.heating_unavailable_period',
    'in.cooling_unavailable_period',
    'in.upgrade_name',
]

str_cols = df_enc.select_dtypes(exclude=[np.number]).columns
drop_const    = [c for c in str_cols if df_enc[c].nunique() <= 1]
drop_explicit = [c for c in DROP_REDUNDANT if c in df_enc.columns]

to_drop = list(set(drop_const + drop_explicit))
df_enc.drop(columns=to_drop, inplace=True)

print(f'Colonnes constantes supprimées  : {len(drop_const)}')
print(f'Colonnes redondantes supprimées : {len(drop_explicit)}')
print(f'Shape : {df_enc.shape}')

## 2. Imputation par groupe (zone climatique)
Doit s'exécuter ici — avant le OHE — pendant que la zone climatique est encore une chaîne.

In [ ]:
GROUPBY_COL = 'in.ashrae_iecc_climate_zone_2004'

num_cols = df_enc.select_dtypes(include=[np.number]).columns
nan_cols = [c for c in num_cols if df_enc[c].isna().any()]

if nan_cols and GROUPBY_COL in df_enc.columns:
    for c in nan_cols:
        global_med  = df_enc[c].median()
        group_meds  = df_enc.groupby(GROUPBY_COL)[c].transform('median')
        df_enc[c]   = df_enc[c].fillna(group_meds.fillna(global_med))
    print(f'Imputation par zone climatique : {len(nan_cols)} colonnes')
    print(f'NaN résiduels : {df_enc[nan_cols].isna().sum().sum()}')
else:
    print('Colonne de groupement absente — ignoré.')

## 3. Groupe 1 — Zone climatique
`in.ashrae_iecc_climate_zone_2004` — 17 zones (1A … 8AK)

In [ ]:
col = 'in.ashrae_iecc_climate_zone_2004'

df_enc[col] = df_enc[col].astype('category')
df_enc = pd.get_dummies(df_enc, columns=[col], prefix='climate', dtype=np.int8, drop_first=True)

print(f'Shape : {df_enc.shape}')

## 4. Groupe 2 — Géométrie

| Colonne | Stratégie |
|---|---|
| `in.orientation` | sin/cos circulaire (8 directions) |
| `in.window_areas` | extraction numérique F/B/L/R (%) |
| `in.geometry_building_number_units_mf/sfa` | midpoint numérique |
| `in.geometry_building_level_mf` | ordinal (Bottom=0, Middle=1, Top=2) |
| `in.geometry_building_horizontal_location_mf/sfa` | OHE drop_first |

In [ ]:
import re

# ── Orientation → sin/cos ────────────────────────────────────────────────────
ORIENT_DEG = {
    'North': 0, 'Northeast': 45, 'East': 90, 'Southeast': 135,
    'South': 180, 'Southwest': 225, 'West': 270, 'Northwest': 315,
}

col = 'in.orientation'
if col in df_enc.columns:
    deg = df_enc[col].map(ORIENT_DEG)
    df_enc['orientation_sin'] = np.sin(np.radians(deg)).astype('float32')
    df_enc['orientation_cos'] = np.cos(np.radians(deg)).astype('float32')
    df_enc.drop(columns=[col], inplace=True)
    print('orientation → sin/cos OK')

# ── Window areas → F/B/L/R numériques ───────────────────────────────────────
def parse_window_areas(val):
    if pd.isna(val):
        return np.nan, np.nan, np.nan, np.nan
    parts = re.findall(r'([FBLR])(\d+)', str(val).upper())
    d = {k: float(v) for k, v in parts}
    return d.get('F', np.nan), d.get('B', np.nan), d.get('L', np.nan), d.get('R', np.nan)

col = 'in.window_areas'
if col in df_enc.columns:
    parsed = df_enc[col].apply(lambda v: pd.Series(parse_window_areas(v),
                                index=['window_front', 'window_back', 'window_left', 'window_right']))
    df_enc = pd.concat([df_enc.drop(columns=[col]), parsed.astype('float32')], axis=1)
    print('window_areas → 4 colonnes OK')

# ── Number of units → midpoint ───────────────────────────────────────────────
def midpoint_units(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s.endswith('+'): return float(s[:-1])
    m = re.match(r'(\d+)-(\d+)', s)
    if m: return (float(m.group(1)) + float(m.group(2))) / 2
    return pd.to_numeric(s, errors='coerce')

for col in ['in.geometry_building_number_units_mf', 'in.geometry_building_number_units_sfa']:
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].apply(midpoint_units)
        print(f'{col} → numérique OK')

# ── Building level → ordinal ─────────────────────────────────────────────────
LEVEL_MAP = {'Bottom Floor': 0, 'Middle Floor': 1, 'Top Floor': 2}
col = 'in.geometry_building_level_mf'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(LEVEL_MAP).astype('Int8')
    print('geometry_building_level_mf → ordinal OK')

# ── Horizontal location → OHE ────────────────────────────────────────────────
HORIZ_PREFIX = {
    'in.geometry_building_horizontal_location_mf':  'horiz_mf',
    'in.geometry_building_horizontal_location_sfa': 'horiz_sfa',
}
horiz_cols = [c for c in HORIZ_PREFIX if c in df_enc.columns]
if horiz_cols:
    df_enc = pd.get_dummies(df_enc, columns=horiz_cols,
                            prefix=[HORIZ_PREFIX[c] for c in horiz_cols],
                            dtype=np.int8, drop_first=True)
    print('horizontal_location OHE OK')

print(f'\nShape : {df_enc.shape}')

## 5. Groupe 3 — Systèmes HVAC
`in.hvac_heating_type`, `in.hvac_cooling_type`, `in.hvac_heating_fuel` — OHE drop_first

In [ ]:
HVAC_COLS = [
    'in.hvac_heating_type',
    'in.hvac_cooling_type',
    'in.hvac_heating_fuel',
]

hvac_cols_present = [c for c in HVAC_COLS if c in df_enc.columns]

for c in hvac_cols_present:
    df_enc[c] = df_enc[c].astype('category')

df_enc = pd.get_dummies(df_enc, columns=hvac_cols_present,
                        prefix=[c.split('in.')[-1] for c in hvac_cols_present],
                        dtype=np.int8, drop_first=True)

print(f'Colonnes HVAC encodées : {hvac_cols_present}')
print(f'Shape : {df_enc.shape}')

## 6. Groupe 4 — Période d'offset consignes (encodage circulaire)

| Colonne | Stratégie |
|---|---|
| `in.cooling_setpoint_offset_period` | direction + period_type + sin/cos heure réelle |
| `in.heating_setpoint_offset_period` | direction + period_type + sin/cos heure réelle |

In [ ]:
DAY_START   = 9
NIGHT_START = 22

def encode_offset_period(val, default_start, with_direction=True):
    if pd.isna(val) or str(val).strip().lower() == 'none':
        base = [0, 0.0, 1.0]
        return [0] + base if with_direction else base

    s = str(val).strip().lower()

    period_type = int('day' in s) + 2 * int('night' in s)
    direction   = int('setup' in s) - int('setback' in s)

    m = re.search(r'([+-]\d+)h', s)
    shift = int(m.group(1)) if m else 0
    actual_hour = (default_start + shift) % 24

    sin_h = float(np.sin(2 * np.pi * actual_hour / 24))
    cos_h = float(np.cos(2 * np.pi * actual_hour / 24))

    base = [period_type, sin_h, cos_h]
    return [direction] + base if with_direction else base


PERIOD_COLS = {
    'in.cooling_setpoint_offset_period': ('cool', DAY_START,   True),
    'in.heating_setpoint_offset_period': ('heat', NIGHT_START, False),
}

for col, (prefix, default_start, with_dir) in PERIOD_COLS.items():
    if col in df_enc.columns:
        names = ([f'{prefix}_direction'] if with_dir else []) + [
            f'{prefix}_period_type', f'{prefix}_start_sin', f'{prefix}_start_cos'
        ]
        encoded = df_enc[col].apply(
            lambda v, ds=default_start, wd=with_dir: pd.Series(
                encode_offset_period(v, ds, wd), index=names
            )
        )
        df_enc = pd.concat([df_enc.drop(columns=[col]), encoded.astype('float32')], axis=1)
        print(f'{col} -> {names}')

print(f'\nShape : {df_enc.shape}')

## Sauvegarde → `features_encodees.parquet`

In [ ]:
out_path = DATA_PROCESSED / 'features_encodees.parquet'
df_enc.to_parquet(out_path, index=False)

print(f'Sauvegardé : {out_path}')
print(f'Shape final : {df_enc.shape}')